# Step 8/9 — Full 2×2 Ablation (C1 × C2) + Conformal

**RESS 2025 — GAN-Conformal-RUL**

The complete ablation over the two training-time contributions, each config also conformalized:

| | real data | + GAN (r=0.75) |
|---|---|---|
| **single-task** | baseline | +GAN |
| **multi-task (λ=1.0)** | +MT | +GAN+MT |

For every config, over five seeds: near-failure and overall RMSE (accuracy) **and** near-failure
PICP/MPIW at 90% (coverage). This answers the synergy question — do C1's coverage benefit and C2's
accuracy benefit stack — and produces the paper's ablation table.

Established so far: C1 improves near-failure *coverage* (0.859→0.885) not accuracy; C2 improves
near-failure *accuracy* (0.244→0.219) not yet tested on coverage. The combined cell settles both.

## 1. Setup

In [ ]:
import os, sys, shutil
os.chdir('/content')
REPO_PATH = '/content/RESS_2025_GAN_Conformal_RUL'
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
!git clone https://github.com/f-khadija-benzine/RESS_2025_GAN_Conformal_RUL.git {REPO_PATH}
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH); sys.path.insert(0, f'{REPO_PATH}/src')
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/ress_checkpoints'
import torch
print('CUDA:', torch.cuda.is_available())

In [ ]:
import numpy as np, pandas as pd
from data_loader import XJTUSYLoader
from health_indicator_v3 import HealthIndicatorPipeline
from windowing import prepare_all_folds, build_folds, WINDOW_SIZE
from model import ModelConfig, RULTrainer, evaluate_grid, average_grids
from multitask import MTConfig, MultiTaskTrainer
from gan import StageGAN, GANConfig
from conformal import calibrate_and_evaluate_mondrian, average_conformal_mondrian, assert_real_calibration

SEEDS = [1, 2, 3, 4, 5]
LAMBDA = 1.0        # chosen from the multi-task sweep
AUG_RATIO = 0.75    # chosen from the augmentation sweep
ALPHA = 0.10
TARGET = 2

## 2. Data + GANs

In [ ]:
CANDIDATES = ['/content/drive/MyDrive/XJTU-SY',
              '/content/drive/MyDrive/XJTU-SY_Bearing_Datasets',
              '/content/drive/MyDrive/data/XJTU-SY']
DATA_ROOT = next((p for p in CANDIDATES if os.path.exists(p)), None); assert DATA_ROOT
all_data = XJTUSYLoader(DATA_ROOT).load_all()
results = HealthIndicatorPipeline(fpt_consecutive=5,
            fpt_min_relative_rise=0.20).process_all(all_data, verbose=False)
fold_data = prepare_all_folds(results, scaling_method='log_clip',
                              rul_target='piecewise', verbose=False)
folds = build_folds(results)
CAL_SIZES = {d['fold']: len(d['y_rul_cal']) for d in fold_data}

def test_bearing_ids(fold):
    bids = []
    for bid in fold['test']:
        n = results[bid]['features'].shape[0]
        bids.extend([bid] * max(0, n - WINDOW_SIZE + 1))
    return np.array(bids)

gans = {}
for k in range(1, 6):
    ck = torch.load(f'{CKPT_DIR}/gan_fold{k}.pt',
                    map_location='cuda' if torch.cuda.is_available() else 'cpu')
    g = StageGAN(GANConfig(**ck['config']))
    g.G.load_state_dict(ck['generator']); g.D.load_state_dict(ck['critic'])
    gans[k] = g
print('data + GANs ready.')

## 3. Helpers — augmentation (training only) and a full pass per config

In [ ]:
def augment_fold(gan, d, ratio, seed):
    rng = np.random.default_rng(seed)
    Xtr, ytr, s = d['X_train'], d['y_rul_train'], d['y_stage_train']
    s3 = s == TARGET; n_real = int(s3.sum()); n_syn = int(round(ratio*n_real))
    if n_syn == 0: return Xtr, ytr, s
    X_syn = gan.sample(n_syn, TARGET)
    y_syn = rng.choice(ytr[s3], size=n_syn, replace=True)
    st_syn = np.full(n_syn, TARGET, dtype=s.dtype)
    return (np.concatenate([Xtr, X_syn.astype(np.float32)]),
            np.concatenate([ytr, y_syn.astype(np.float32)]),
            np.concatenate([s, st_syn]))

def run_pass(d, f, seed, use_gan, use_mt):
    """Train one fold under a config; return (grid, conformal) both real-calibrated."""
    k = d['fold']
    if use_gan:
        Xtr, ytr, str_ = augment_fold(gans[k], d, AUG_RATIO, seed)
    else:
        Xtr, ytr, str_ = d['X_train'], d['y_rul_train'], d['y_stage_train']

    if use_mt:
        cfg = MTConfig(epochs=100, patience=25, lr=5e-4, seed=seed, lambda_stage=LAMBDA)
        tr = MultiTaskTrainer(cfg)
        tr.fit(Xtr, ytr, str_, d['X_val'], d['y_rul_val'], verbose=False)
        yp_cal = tr.predict_rul(d['X_cal']); yp_test = tr.predict_rul(d['X_test'])
    else:
        cfg = ModelConfig(epochs=100, patience=25, lr=5e-4, seed=seed)
        tr = RULTrainer(cfg)
        tr.fit(Xtr, ytr, d['X_val'], d['y_rul_val'],
               stage_train=str_, stage_val=d['y_stage_val'],
               mask_healthy=False, verbose=False)
        yp_cal = tr.predict(d['X_cal']); yp_test = tr.predict(d['X_test'])

    assert_real_calibration(CAL_SIZES[k], d['y_rul_cal'])   # guard: no synth in cal
    grid = evaluate_grid(d['y_rul_test'], yp_test, d['y_stage_test'], test_bearing_ids(f))
    conf = calibrate_and_evaluate_mondrian(
        d['y_rul_cal'], yp_cal, d['y_stage_cal'],
        d['y_rul_test'], yp_test, d['y_stage_test'], alpha=ALPHA)
    return grid, conf

## 4. Run all four configs, multi-seed

In [ ]:
CONFIGS = {
    'baseline':  (False, False),
    '+GAN':      (True,  False),
    '+MT':       (False, True),
    '+GAN+MT':   (True,  True),
}

results_acc = {c: [] for c in CONFIGS}   # per-seed averaged grids
results_cov = {c: [] for c in CONFIGS}   # per-seed averaged conformal

for name, (use_gan, use_mt) in CONFIGS.items():
    print(f'\n=== {name} ===')
    for seed in SEEDS:
        grids, confs = [], []
        for d, f in zip(fold_data, folds):
            g, c = run_pass(d, f, seed, use_gan, use_mt)
            grids.append(g); confs.append(c)
        results_acc[name].append(average_grids(grids))
        results_cov[name].append(average_conformal_mondrian(confs))
        print(f'  seed {seed} done')

## 5. Ablation table — accuracy and coverage together

In [ ]:
def ms(seeds, sub, metric):
    v = [s[sub][metric] for s in seeds]; return np.mean(v), np.std(v)

print(f"{'config':10s} {'nf RMSE':>16s} {'nf PICP':>16s} {'nf MPIW':>16s}")
print('-'*62)
for name in CONFIGS:
    rm, rs = ms(results_acc[name], 'nearfail', 'per_bearing')
    pm, ps = ms(results_cov[name], 'nearfail', 'picp')
    wm, ws = ms(results_cov[name], 'nearfail', 'mpiw')
    print(f"{name:10s} {rm:6.4f}±{rs:.4f}   {pm:6.3f}±{ps:.3f}   {wm:6.4f}±{ws:.4f}")

print('\nRead: +GAN → coverage; +MT → accuracy; +GAN+MT → do both stack?')

In [ ]:
# Save the ablation to Drive so it survives the session
import json
summary = {}
for name in CONFIGS:
    summary[name] = {
        'nf_rmse': ms(results_acc[name], 'nearfail', 'per_bearing'),
        'overall_rmse': ms(results_acc[name], 'overall', 'per_bearing'),
        'nf_picp': ms(results_cov[name], 'nearfail', 'picp'),
        'nf_mpiw': ms(results_cov[name], 'nearfail', 'mpiw'),
    }
with open(f'{CKPT_DIR}/ablation_summary.json', 'w') as fh:
    json.dump({k: {m: list(v) for m, v in d.items()} for k, d in summary.items()}, fh, indent=2)
print('saved ablation_summary.json to Drive')

## Reading the synergy

The four rows answer the paper's central claim:

- **+GAN vs baseline** — coverage improves (0.86→0.89), accuracy flat. *C1 = calibration.*
- **+MT vs baseline** — accuracy improves (0.244→0.219), coverage to be read. *C2 = accuracy.*
- **+GAN+MT vs each** — the synergy. If accuracy matches +MT *and* coverage matches +GAN, the two
  contributions **stack**: the combined model is both more accurate and better-calibrated in the
  near-failure regime than any single-contribution variant. If they interfere, that is reported
  honestly. Either way this is the headline ablation.

Conformal coverage is computed on real calibration throughout (asserted); synthetic data enters
training only, in every augmented config.